In [1]:
import ssl, urllib.request, pandas as pd
from pathlib import Path

ctx = ssl._create_unverified_context()
OUT_DIR = Path("../public")

def fetch_fred(series_id, col_name):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    print(f"Fetching {series_id}...")
    with urllib.request.urlopen(url, context=ctx) as r:
        raw = pd.read_csv(r)
    raw.columns = ["date", col_name]
    raw["year"] = pd.to_datetime(raw["date"]).dt.year
    raw[col_name] = pd.to_numeric(raw[col_name], errors="coerce")
    return raw.dropna(subset=[col_name])

# Real interest rate — monthly, need annual average
real_rate = fetch_fred("REAINTRATREARAT10Y", "real_rate_10y")
real_rate_annual = real_rate.groupby("year")["real_rate_10y"].mean().reset_index()

# Private investment % GDP — quarterly, need annual average
priv_inv = fetch_fred("A006RE1Q156NBEA", "private_investment_pct_gdp")
priv_inv_annual = priv_inv.groupby("year")["private_investment_pct_gdp"].mean().reset_index()

# Deficit % GDP — already have this but fetch fresh for alignment
deficit = fetch_fred("FYFSGDA188S", "deficit_pct_gdp")
deficit_annual = deficit.groupby("year")["deficit_pct_gdp"].mean().reset_index()

# Merge on year
df = deficit_annual.merge(real_rate_annual, on="year", how="inner")
df = df.merge(priv_inv_annual, on="year", how="inner")
df = df.sort_values("year").reset_index(drop=True)

print(f"Coverage: {df['year'].min()} – {df['year'].max()}  ({len(df)} rows)")
print(df.tail(8).to_string(index=False))

Fetching REAINTRATREARAT10Y...
Fetching A006RE1Q156NBEA...
Fetching FYFSGDA188S...
Coverage: 1982 – 2025  (44 rows)
 year  deficit_pct_gdp  real_rate_10y  private_investment_pct_gdp
 2018         -3.77157       1.156195                      18.025
 2019         -4.56634       0.534974                      18.050
 2020        -14.65457      -0.119478                      17.575
 2021        -11.69768      -0.031760                      17.875
 2022         -5.28091       1.059399                      18.625
 2023         -6.09546       1.657494                      18.050
 2024         -6.20107       1.818494                      17.950
 2025         -5.77031       1.713265                      17.750


In [2]:
# Sanity checks
checks = {
    1983: {"deficit_pct_gdp": (-7, -5), "real_rate_10y": (4, 7)},    # Reagan — big deficit, high real rates
    2000: {"deficit_pct_gdp": (1, 3),   "real_rate_10y": (2, 5)},    # Clinton surplus
    2009: {"deficit_pct_gdp": (-11, -8),"real_rate_10y": (-1, 2)},   # GFC — big deficit but Fed suppressed rates
}

all_ok = True
for year, cols in checks.items():
    row = df[df["year"] == year]
    if row.empty:
        print(f"  MISSING {year}"); all_ok = False; continue
    for col, (lo, hi) in cols.items():
        val = row[col].iloc[0]
        ok = lo <= val <= hi
        print(f"  {'OK  ' if ok else 'FAIL'}  {year} {col}: {val:.2f}  (expected {lo} to {hi})")
        if not ok: all_ok = False

print()
print("All checks passed ✓" if all_ok else "Some checks failed.")

  OK    1983 deficit_pct_gdp: -5.72  (expected -7 to -5)
  OK    1983 real_rate_10y: 5.89  (expected 4 to 7)
  OK    2000 deficit_pct_gdp: 2.30  (expected 1 to 3)
  OK    2000 real_rate_10y: 3.12  (expected 2 to 5)
  OK    2009 deficit_pct_gdp: -9.76  (expected -11 to -8)
  OK    2009 real_rate_10y: 1.05  (expected -1 to 2)

All checks passed ✓


In [3]:
out = OUT_DIR / "crowding_out.csv"
df.to_csv(out, index=False)
print(f"Written {len(df)} rows → {out}")

Written 44 rows → ../public/crowding_out.csv


In [6]:
# Japan debt % GDP — combine actual (to 2023) + IMF projections (2024+)
japan_debt_actual = fetch_fred("GGGDTAJPA188N", "japan_debt_pct_gdp")
japan_debt_proj   = fetch_fred("GGGDTPJPA188N", "japan_debt_pct_gdp")

debt_actual_annual = japan_debt_actual.groupby("year")["japan_debt_pct_gdp"].mean().reset_index()
debt_proj_annual   = japan_debt_proj.groupby("year")["japan_debt_pct_gdp"].mean().reset_index()
debt_proj_annual   = debt_proj_annual[debt_proj_annual["year"] >= 2024]
japan_debt_annual  = pd.concat([debt_actual_annual, debt_proj_annual]).reset_index(drop=True)

# Japan 10-year bond yield — monthly, need annual average
japan_yield = fetch_fred("IRLTLT01JPM156N", "japan_10y_yield")
japan_yield_annual = japan_yield.groupby("year")["japan_10y_yield"].mean().reset_index()

# Japan real GDP growth — compute pct change from level series
japan_gdp_raw = fetch_fred("JPNRGDPEXP", "gdp_level")
japan_gdp_annual = japan_gdp_raw.groupby("year")["gdp_level"].mean().reset_index()
japan_gdp_annual["japan_real_gdp_growth"] = japan_gdp_annual["gdp_level"].pct_change() * 100
japan_gdp_annual = japan_gdp_annual.drop(columns=["gdp_level"]).dropna()

# Merge
df = japan_debt_annual.merge(japan_yield_annual, on="year", how="outer")
df = df.merge(japan_gdp_annual, on="year", how="outer")
df = df.sort_values("year").reset_index(drop=True)
print(f"Coverage: {df['year'].min()} – {df['year'].max()}")
print(df.tail(10).to_string(index=False))

Fetching GGGDTAJPA188N...
Fetching GGGDTPJPA188N...
Fetching IRLTLT01JPM156N...
Fetching JPNRGDPEXP...
Coverage: 1980 – 2030
 year  japan_debt_pct_gdp  japan_10y_yield  japan_real_gdp_growth
 2021             253.652         0.071667               3.655908
 2022             248.309         0.231667               1.290878
 2023             239.971         0.562500               0.716573
 2024             236.660         0.918333              -0.232818
 2025             234.858         1.553333               1.192926
 2026             233.730         2.175000                    NaN
 2027             232.099              NaN                    NaN
 2028             231.212              NaN                    NaN
 2029             231.085              NaN                    NaN
 2030             231.676              NaN                    NaN


In [7]:
# Sanity checks
checks = {
    1995: {"japan_debt_pct_gdp": (60, 90),   "japan_10y_yield": (2, 4)},
    2010: {"japan_debt_pct_gdp": (180, 220),  "japan_10y_yield": (0.5, 2)},
    2020: {"japan_debt_pct_gdp": (230, 270),  "japan_10y_yield": (-0.5, 1)},
    2024: {"japan_debt_pct_gdp": (220, 270),  "japan_10y_yield": (0.5, 2)},  # rates rising
}

all_ok = True
for year, cols in checks.items():
    row = df[df["year"] == year]
    if row.empty:
        print(f"  MISSING {year}"); all_ok = False; continue
    for col, (lo, hi) in cols.items():
        val = row[col].iloc[0]
        ok = lo <= val <= hi
        print(f"  {'OK  ' if ok else 'FAIL'}  {year} {col}: {val:.2f}  (expected {lo} to {hi})")
        if not ok: all_ok = False

print()
print("All checks passed ✓" if all_ok else "Some checks failed.")

  FAIL  1995 japan_debt_pct_gdp: 92.53  (expected 60 to 90)
  OK    1995 japan_10y_yield: 3.44  (expected 2 to 4)
  OK    2010 japan_debt_pct_gdp: 205.88  (expected 180 to 220)
  OK    2010 japan_10y_yield: 1.15  (expected 0.5 to 2)
  OK    2020 japan_debt_pct_gdp: 258.37  (expected 230 to 270)
  OK    2020 japan_10y_yield: -0.01  (expected -0.5 to 1)
  OK    2024 japan_debt_pct_gdp: 236.66  (expected 220 to 270)
  OK    2024 japan_10y_yield: 0.92  (expected 0.5 to 2)

Some checks failed.


In [8]:
out = OUT_DIR / "japan_case_study.csv"
df.to_csv(out, index=False)
print(f"Written {len(df)} rows → {out}")

Written 51 rows → ../public/japan_case_study.csv
